# Marketing OS — render job (manual, Colab gratuito)

Uso interativo, iniciado por você. **Não** é servidor permanente: o notebook faz um render e termina. CPU basta (nenhuma GPU é exigida). Assets privados de marca só entram por upload dentro desta sessão e somem quando ela fecha; nada é enviado ao repositório público.

1. Cole o job spec impresso por `node ops/cli.ts dispatch <job> --prefer colab` na célula abaixo.
2. Execute tudo.
3. Baixe o MP4 e registre o hash local com `node ops/cli.ts complete <job> --sha256 <hash>`.

In [ ]:
JOB = {}  # cole aqui: {"job_id": "job-...", "spec": {...}}

In [ ]:
%%bash
set -e
curl -fsSL https://deb.nodesource.com/setup_24.x | bash - >/dev/null
apt-get install -y -qq nodejs ffmpeg >/dev/null
npm i -g pnpm@11.25.0 >/dev/null
rm -rf marketing-os && git clone --depth 1 https://github.com/DarlysTiphereth/marketing-os.git
cd marketing-os/experiments/remotion && pnpm install --frozen-lockfile >/dev/null && echo ready

In [ ]:
# Opcional: assets privados (apenas nesta sessão). Coloque-os nos caminhos que o creative.json espera.
# from google.colab import files; up = files.upload()

In [ ]:
import json, os, subprocess
assert JOB.get('job_id','').startswith('job-'), 'cole o job spec primeiro'
os.makedirs('marketing-os/ops/jobs', exist_ok=True)
json.dump(JOB, open(f"marketing-os/ops/jobs/{JOB['job_id']}.json", 'w'))
env = dict(os.environ, MOS_HOME='/content/mos', MOS_FFMPEG='ffmpeg', MOS_FFPROBE='ffprobe')
subprocess.run(['node', 'ops/cli.ts', 'run-here', f"ops/jobs/{JOB['job_id']}.json"], cwd='marketing-os', env=env, check=True)

In [ ]:
import json, glob
s = json.load(open(f"/content/mos/runs/{JOB['job_id']}.json"))
print('sha256:', s['output_sha256'], 'QC:', s['qc_verdict'])
from google.colab import files
files.download(glob.glob('/content/mos/runs-out/**/final.mp4', recursive=True)[0])